# Model Experiment — `{MODEL_NAME}`

**Copy this template** to `model_experiment_{MODEL_NAME}.ipynb` and fill in the model.

MLflow structure required by the assignment:
- Experiment: **`{MODEL_NAME}_Training`**
- Runs inside it: `{MODEL_NAME}_Cleaning`, `{MODEL_NAME}_Feature_Selection`, `{MODEL_NAME}_CV`, `{MODEL_NAME}_Final`.

The best model of this architecture is saved as a **sklearn Pipeline** that runs on the **raw** test set, and logged to MLflow so `model_inference.ipynb` can load it.

In [ ]:
# === Bootstrap: make the shared `src` package importable (local + Kaggle) ===
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/sophyrise/walmart-sales-forecasting.git"

if Path("/kaggle/input").exists():
    # On Kaggle: clone the repo to pull in the shared src/ package.
    dst = Path("/kaggle/working/walmart-sales-forecasting")
    if not dst.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dst)], check=True)
    sys.path.insert(0, str(dst))
else:
    # Local: notebooks/ sits one level under the repo root.
    sys.path.insert(0, str(Path.cwd().parent))

from src import config
print("On Kaggle:", config.ON_KAGGLE, "| raw data dir:", config.RAW_DIR)

In [ ]:
MODEL_NAME = 'TEMPLATE'   # <-- e.g. 'XGBoost'
EXPERIMENT = f'{MODEL_NAME}_Training'

In [ ]:
import numpy as np, pandas as pd, mlflow
from src import data_loader as dl, features as ft, validation as val
from src.metrics import wmae, weights_from_holiday
from src.pipeline import build_pipeline
from src.tracking import init_tracking

uri = init_tracking(EXPERIMENT)
print('MLflow ->', uri, '| experiment:', EXPERIMENT)

## Load raw data

In [ ]:
raw_train = dl.load_raw('train')
raw_test  = dl.load_raw('test')
raw_train.shape, raw_test.shape

## Run 1 — Cleaning
Log data-cleaning decisions (missing values, negatives, dtypes) as an MLflow run.

In [ ]:
with mlflow.start_run(run_name=f'{MODEL_NAME}_Cleaning'):
    mlflow.log_param('markdown_impute', 'fill_0')
    mlflow.log_param('clip_negatives_in_submission', True)
    mlflow.log_metric('n_train_rows', len(raw_train))
    mlflow.log_metric('n_negative_sales', int((raw_train.Weekly_Sales < 0).sum()))
    print('logged cleaning run')

## Run 2 — Feature Selection
Build features and record which columns feed the model.

In [ ]:
feat_train = ft.build_features(dl.load_merged('train'), add_lags=False)
feature_cols = ft.feature_columns(feat_train)
with mlflow.start_run(run_name=f'{MODEL_NAME}_Feature_Selection'):
    mlflow.log_param('n_features', len(feature_cols))
    mlflow.log_param('features', feature_cols)
print(len(feature_cols), 'features'); feature_cols

## Run 3 — Cross-validation
Expanding-window backtest with the competition metric (WMAE).

In [ ]:
# TODO: replace with your architecture's estimator
from sklearn.tree import DecisionTreeRegressor
def make_model():
    return DecisionTreeRegressor(max_depth=10, random_state=config_seed())
def config_seed():
    from src import config; return config.RANDOM_SEED

scores = []
with mlflow.start_run(run_name=f'{MODEL_NAME}_CV'):
    for k, (tr_idx, va_idx) in enumerate(val.expanding_splits(raw_train, n_splits=3, horizon=8)):
        tr, va = raw_train.iloc[tr_idx], raw_train.iloc[va_idx]
        pipe = build_pipeline(make_model())
        pipe.fit(tr, tr['Weekly_Sales'])
        pred = pipe.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        scores.append(score)
        mlflow.log_metric('fold_wmae', score, step=k)
        print(f'fold {k}: WMAE={score:,.1f}')
    mlflow.log_metric('cv_wmae_mean', float(np.mean(scores)))
    mlflow.log_metric('cv_wmae_std', float(np.std(scores)))
print('CV WMAE:', np.mean(scores))

## Run 4 — Final fit + log Pipeline
Refit on all data and log the **Pipeline** (runs on raw test) as an MLflow model artifact.

In [ ]:
from mlflow.models import infer_signature
final_pipe = build_pipeline(make_model())
final_pipe.fit(raw_train, raw_train['Weekly_Sales'])
example = raw_test.head(5)
sig = infer_signature(example, final_pipe.predict(example))
with mlflow.start_run(run_name=f'{MODEL_NAME}_Final') as run:
    holdout_tr, holdout_va = val.time_holdout(raw_train, weeks=8)
    p = build_pipeline(make_model())
    p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
    hv = raw_train.iloc[holdout_va]
    holdout_wmae = wmae(hv['Weekly_Sales'], p.predict(hv), hv['IsHoliday'])
    mlflow.log_metric('holdout_wmae', holdout_wmae)
    mlflow.sklearn.log_model(final_pipe, artifact_path='pipeline',
                             signature=sig, input_example=example)
    print('holdout WMAE:', holdout_wmae, '| run_id:', run.info.run_id)

> **Registering the best model:** once you know this architecture is the overall best, register it from the Final run — either via the MLflow UI or:
> ```python
> mlflow.register_model(f'runs:/{run.info.run_id}/pipeline', 'walmart_best_model')
> ```